Ноутбук кодирует русский корпус моделью E. На выходе сохраняются эмбеддинги документов, которые потом используются для оценки метрик.


In [ ]:
!nvidia-smi

import os, json, glob, hashlib, gc, time
from collections import Counter
import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-2-7b-hf"

MAIN_INPUT_PATH = "/kaggle/input/datasets/sukiss/prmptr/tevatron_ru_promptriever_train (2).jsonl"

TEST_CHUNKS_PATH = "/kaggle/input/datasets/sukiss/test-dataset/chunks_testset.jsonl"

USE_ADAPTER = True
ADAPTER_DIR = "/kaggle/input/datasets/sukiss/adapters"

OUT_DIR = "/kaggle/working/proxy_encoding"
os.makedirs(OUT_DIR, exist_ok=True)

BATCH_DOCS = 16
PASSAGE_MAX_LEN = 128
DTYPE_SAVE = np.float16

def sha1_id(text: str) -> str:
    return hashlib.sha1(text.strip().encode("utf-8")).hexdigest()

def write_corpus_jsonl(docid_to_text: dict, out_jsonl: str):
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for did, text in docid_to_text.items():
            f.write(json.dumps({"docid": did, "text": text}, ensure_ascii=False) + "\n")

def parse_docs_from_main(path: str):
    """
    Supports two formats:
    1) rusmsmarco_20k_train.jsonl style:
       {"query_id","query","positive_passages":[{"text","docid"?}], "negative_passages":[{"text","docid"?},...]}
    2) tevatron_ru_promptriever_train.jsonl style:
       {"query_id","query","positive_passages":[...], "negative_passages":[...], "metadata":{...}}
    We extract ALL passage texts we see.
    """
    docs = {}
    stats = Counter()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            stats["rows_seen"] += 1
            try:
                item = json.loads(line)
            except Exception:
                stats["bad_json"] += 1
                continue

            for key in ["positive_passages", "negative_passages"]:
                for p in item.get(key, []) or []:
                    text = (p.get("text") or "").strip()
                    if not text:
                        stats["empty_text"] += 1
                        continue
                    did = str(p.get("docid") or "") or sha1_id(text)
                    if did in docs:
                        stats["dup_docid"] += 1
                    else:
                        docs[did] = text
                        stats["docs_added"] += 1
    return docs, stats

def parse_docs_from_chunks(path: str):
    """
    chunks_testset.jsonl style (from your Colab joiner):
      {"query_id","positive","original_negs":[...],"generated_negs":[...], ...}
    """
    docs = {}
    stats = Counter()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            stats["rows_seen"] += 1
            item = json.loads(line)

            texts = []
            texts.append((item.get("positive") or "").strip())
            texts += [(x or "").strip() for x in (item.get("original_negs") or [])[:2]]
            texts += [(x or "").strip() for x in (item.get("generated_negs") or [])[:3]]

            for text in texts:
                if not text:
                    stats["empty_text"] += 1
                    continue
                did = sha1_id(text)
                if did in docs:
                    stats["dup_docid"] += 1
                else:
                    docs[did] = text
                    stats["docs_added"] += 1
    return docs, stats

def exists_adapter(path):
    return os.path.exists(os.path.join(path, "adapter_config.json")) and os.path.exists(os.path.join(path, "adapter_model.safetensors"))

assert os.path.exists(MAIN_INPUT_PATH), f"Missing MAIN_INPUT_PATH: {MAIN_INPUT_PATH}"
assert os.path.exists(TEST_CHUNKS_PATH), f"Missing TEST_CHUNKS_PATH: {TEST_CHUNKS_PATH}"
if USE_ADAPTER:
    assert exists_adapter(ADAPTER_DIR), f"Missing adapter files in ADAPTER_DIR: {ADAPTER_DIR}"

if HF_TOKEN and HF_TOKEN != "PASTE_HF_TOKEN_HERE":
    if HF_TOKEN:
    if HF_TOKEN:
    login(token=HF_TOKEN)

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tok_src = ADAPTER_DIR if USE_ADAPTER else BASE_MODEL
tok = AutoTokenizer.from_pretrained(tok_src, use_fast=False, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

base = AutoModel.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
base.config.use_cache = False

model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False) if USE_ADAPTER else base
model.eval()
print("Loaded model. Adapter:", USE_ADAPTER, "device:", next(model.parameters()).device)

def add_eos(texts): return [t + tok.eos_token for t in texts]
def eos_pool(last_hidden_state, attention_mask):
    lengths = attention_mask.sum(dim=1) - 1
    idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)
    return last_hidden_state[idx, lengths]

@torch.inference_mode()
def encode_texts(texts, max_len):
    batch = tok(add_eos(texts), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    batch = {k: v.to(model.device) for k, v in batch.items()}
    out = model(**batch)
    emb = eos_pool(out.last_hidden_state, batch["attention_mask"])
    return F.normalize(emb, p=2, dim=-1)

def encode_corpus(docid_to_text: dict, name: str):
    out_jsonl = os.path.join(OUT_DIR, f"{name}_corpus.jsonl")
    out_docids = os.path.join(OUT_DIR, f"{name}_docids.txt")
    out_emb = os.path.join(OUT_DIR, f"{name}_embeddings.npy")

    docids = list(docid_to_text.keys())
    texts = [docid_to_text[d] for d in docids]

    print(f"\n[{name}] docs:", len(docids))
    write_corpus_jsonl(docid_to_text, out_jsonl)

    all_emb = np.memmap(out_emb, dtype=DTYPE_SAVE, mode="w+", shape=(len(docids), 4096))
    first = encode_texts(texts[:1], PASSAGE_MAX_LEN).detach().cpu().numpy().astype(DTYPE_SAVE)
    dim = first.shape[1]
    del all_emb
    os.remove(out_emb)

    emb = np.memmap(out_emb, dtype=DTYPE_SAVE, mode="w+", shape=(len(docids), dim))
    emb[0:1] = first

    start = time.time()
    i = 1
    while i < len(texts):
        j = min(len(texts), i + BATCH_DOCS)
        batch_emb = encode_texts(texts[i:j], PASSAGE_MAX_LEN).detach().cpu().numpy().astype(DTYPE_SAVE)
        emb[i:j] = batch_emb
        i = j
        if i % 500 == 0:
            print(f"[{name}] encoded {i}/{len(texts)}")
    emb.flush()
    dt = time.time() - start

    with open(out_docids, "w", encoding="utf-8") as f:
        for d in docids:
            f.write(d + "\n")

    print(f"[{name}] saved:")
    print(" corpus:", out_jsonl, "MB", round(os.path.getsize(out_jsonl)/1024/1024, 3))
    print(" docids:", out_docids, "MB", round(os.path.getsize(out_docids)/1024/1024, 3))
    print(" emb   :", out_emb, "MB", round(os.path.getsize(out_emb)/1024/1024, 3), "dim", dim, "time_s", round(dt, 1))

main_docs, main_stats = parse_docs_from_main(MAIN_INPUT_PATH)
print("Main stats:", main_stats)

test_docs, test_stats = parse_docs_from_chunks(TEST_CHUNKS_PATH)
print("Test stats:", test_stats)

encode_corpus(main_docs, "main")
encode_corpus(test_docs, "test")

print("\nDONE. Outputs in:", OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))
